<a href="https://colab.research.google.com/github/Abdulkareem300/hypertension-predictor/blob/main/HYPERTENSION_STAGE_PREDICTOR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# systolic and diastolic blood pressure using hypertension dataset


In [ ]:
import pandas as pd
df=pd.read_csv("/content/Hypertension data set with lebeld by stage.csv")

In [ ]:
print(df.head()) #checking the top 5 of the dataset
print(df.info()) #to give the information of the dataset
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.describe().T

In [ ]:
print(df.duplicated().sum()) #to check the number of duplicates

In [ ]:
print(f"Shape before dropping duplicates: {df.shape}")
df = df.drop_duplicates()
print(f"Shape after dropping duplicates: {df.shape}")

In [ ]:
# Section: Exploratory Data Analysis

### Multi-Output Classification Model for SBP and DBP

Since both `SBP` and `DBP` are categorical/ordinal values in this dataset, we can train a multi-output classifier to predict both values at the same time using features such as age, BMI, sex, lifestyle, and kidney function markers.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="darkgrid")

# Creating a 2 row, 3 column grid with adjusted dimensions
fig, axes = plt.subplots(2, 3, figsize=(20, 13))

# --- ROW 1: Systolic Blood Pressure (SBP) Plots ---

# 1. Distribution of SBP against Age
sns.countplot(data=df, x="age", hue="SBP", ax=axes[0, 0], palette="Blues")
axes[0, 0].set_title("Distribution of SBP against Age")
axes[0, 0].set_xlabel("Age Category")
axes[0, 0].set_ylabel("Patient Count")
axes[0, 0].legend(title="SBP Level")

# 2. Distribution of SBP against BMI
sns.countplot(data=df, x="BMI", hue="SBP", ax=axes[0, 1], palette="YlOrBr")
axes[0, 1].set_title("Distribution of SBP against BMI")
axes[0, 1].set_xlabel("BMI Category")
axes[0, 1].set_ylabel("Patient Count")
axes[0, 1].legend(title="SBP Level")

# 3. Distribution of SBP against Sex
sns.countplot(data=df, x="sex", hue="SBP", ax=axes[0, 2], palette="Greens")
axes[0, 2].set_title("Distribution of SBP against Sex")
axes[0, 2].set_xlabel("Sex (0 = Female, 1 = Male)")
axes[0, 2].set_ylabel("Patient Count")
axes[0, 2].legend(title="SBP Level")

# --- ROW 2: Diastolic Blood Pressure (DBP) Plots ---

# 4. Distribution of DBP against Age
sns.countplot(data=df, x="age", hue="DBP", ax=axes[1, 0], palette="Reds")
axes[1, 0].set_title("Distribution of DBP against Age")
axes[1, 0].set_xlabel("Age Category")
axes[1, 0].set_ylabel("Patient Count")
axes[1, 0].legend(title="DBP Level")

# 5. Distribution of DBP against BMI
sns.countplot(data=df, x="BMI", hue="DBP", ax=axes[1, 1], palette="Purples")
axes[1, 1].set_title("Distribution of DBP against BMI")
axes[1, 1].set_xlabel("BMI Category (0 to 3)")
axes[1, 1].set_ylabel("Patient Count")
axes[1, 1].legend(title="DBP Level")

# 6. Distribution of DBP against Sex
sns.countplot(data=df, x="sex", hue="DBP", ax=axes[1, 2], palette="Oranges")
axes[1, 2].set_title("Distribution of DBP against Sex")
axes[1, 2].set_xlabel("Sex (0 = Female, 1 = Male)")
axes[1, 2].set_ylabel("Patient Count")
axes[1, 2].legend(title="DBP Level")

# Set customizable gap spacing between the plots
plt.subplots_adjust(hspace=0.4, wspace=0.3)

plt.show()

In [ ]:
# Section: Feature Engineering and Feature Scaling

```markdown
## Feature Engineering, Target Selection, and Scaling
We will define our target variables as `SBP`, `DBP`, and the hypertension classification `level`. Our features will include demographic data, lifestyle indices, and clinical markers.
```

In [ ]:
# note: before training the model ensure you define the features and the label

from sklearn.model_selection import train_test_split #to train-test split the model
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report

# 1. Define Features (X) and Targets (y)
# We exclude the targets from X. We include 'level', 'SBP', and 'DBP' as targets.
target_cols = ['SBP', 'DBP', 'level']
feature_cols = [col for col in df.columns if col not in target_cols]

X = df[feature_cols]
y = df[target_cols]

# 2. Split into Train and Test Sets (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Scale Features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Feature matrix shape: {X_train_scaled.shape}")
print(f"Target matrix shape: {y_train.shape}")
print("Target columns being predicted simultaneously:", list(y.columns))

In [ ]:
# Convert targets to strings to avoid numpy mixed-type comparison errors (int vs str)
y_train_str = y_train.astype(str)
y_test_str = y_test.astype(str)

# 1. Initialize the Base Classifier
base_forest = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')

# 2. Wrap it in a MultiOutputClassifier
multi_target_forest = MultiOutputClassifier(base_forest, n_jobs=-1)

# 3. Train the Model using string targets
multi_target_forest.fit(X_train_scaled, y_train_str)

# 4. Predict on Test Data
y_pred = multi_target_forest.predict(X_test_scaled)

# 5. Evaluate the results for each target output separately
for i, col in enumerate(target_cols):
    print(f"===================================================")
    print(f" CLASSIFICATION REPORT FOR TARGET: {col.upper()} ")
    print(f"===================================================")
    # y_pred is a numpy array of shape (n_samples, n_targets)
    print(classification_report(y_test_str[col], y_pred[:, i]))
    print("\n")

### Step 1: Save the Trained Model and Scaler
We need to export our trained `multi_target_forest` model and our fitted `scaler` using `joblib` so that they can be reused inside the Streamlit web application.

In [ ]:
import joblib

# Save the model and scaler to disk
joblib.dump(multi_target_forest, 'hypertension_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

print("Saved model and scaler successfully!")

### Step 2: Create the Streamlit App Script (`app.py`)
This block writes a `app.py` script containing the User Interface and prediction logic. It accepts your 9 clinical features as inputs, scales them, runs them through the loaded model, and returns predicted levels for **SBP**, **DBP**, and **Hypertension Stage Classification** simultaneously.

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib

# 1. Load the model and scaler
@st.cache_resource
def load_assets():
    model = joblib.load('hypertension_model.pkl')
    scaler = joblib.load('scaler.pkl')
    return model, scaler

model, scaler = load_assets()

# 2. Design the Streamlit UI
st.title("🩺 Hypertension Stage Predictor")
st.write("Input the demographic, lifestyle, and clinical parameters below to predict SBP, DBP, and classification level simultaneously.")

# Create a structured layout with columns
col1, col2, col3 = st.columns(3)

with col1:
    age = st.selectbox("Age Category", [0, 1, 2], help="Choose your age category index")
    sex = st.selectbox("Sex", [0, 1], format_func=lambda x: "Male" if x == 1 else "Female")
    BMI = st.slider("BMI Category", 0, 3, 1)

with col2:
    resi = st.selectbox("Residence", [0, 1], format_func=lambda x: "Urban" if x == 1 else "Rural")
    smoking = st.selectbox("Smoking Status", [0, 1], format_func=lambda x: "Yes" if x == 1 else "No")
    odisease = st.slider("Other Diseases Index", 0, 3, 0)

with col3:
    creantine = st.selectbox("Creatinine level high?", [0, 1], format_func=lambda x: "Yes" if x == 1 else "No")
    bun = st.selectbox("BUN level high?", [0, 1], format_func=lambda x: "Yes" if x == 1 else "No")
    noofmed = st.selectbox("Number of Meds", [0, 1])

# 3. Model Prediction logic
if st.button("Predict Hypertension Stages", type="primary"):
    # Assemble inputs in correct feature sequence
    input_data = pd.DataFrame([[
        age, sex, BMI, resi, smoking, odisease, creantine, bun, noofmed
    ]], columns=['age', 'sex', 'BMI', 'Resi', 'Smoking', 'odisease', 'creantine', 'BUN', 'Noofmed'])

    # Scale features
    input_scaled = scaler.transform(input_data)

    # Predict
    prediction = model.predict(input_scaled)[0]

    # Render results nicely
    st.success("### Prediction Results:")
    st.metric(label="Predicted Systolic Blood Pressure Category (SBP)", value=prediction[0])
    st.metric(label="Predicted Diastolic Blood Pressure Category (DBP)", value=prediction[1])
    st.metric(label="Hypertension Class Level", value=prediction[2])

In [ ]:
# 1. Run the Streamlit app in the background
# 2. Expose port 8501 using localtunnel to get a shareable URL
import urllib
print("Your password/tunnel IP for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

!streamlit run app.py & npx localtunnel --port 8501

In [25]:
%%writefile requirements.txt
streamlit
pandas
numpy
scikit-learn
joblib

Writing requirements.txt


In [26]:
# Run streamlit in the background and expose it using localtunnel
!nohup streamlit run app.py &
!npx localtunnel --port 8501

nohup: appending output to 'nohup.out'
⠙⠹⠸⠼⠴⠦⠧your url is: https://better-rockets-enter.loca.lt
^C
